[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/conic_sections_course.ipynb)

# Conic sections full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and three pictures you can move
yourself: a cone you slice, an ellipse drawn with two pins and a string, and one slider that turns an ellipse into a
parabola and then a hyperbola. Work each question out on paper first, then type your answer into the box and press
**check**. It tells you whether you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Fractions as `3/5`, roots as `sqrt(5)`, minus as `-`. For several answers, separate them with
commas, in the order the question asks. When a question asks for an equation, type it in any form you like, such as
`x^2/16 + y^2/12 = 1` or `3x^2 + 4y^2 = 48`: the check knows they're the same curve.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re
from fractions import Fraction as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"     # the plane and c · distances and b · the answer and a

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x, y = sp.symbols("x y")
r = sp.Rational
NAMES = {"x": x, "y": y, "pi": sp.pi, "sqrt": sp.sqrt}

def prep(s):
    """Typed symbols → Python: − × ÷ ² √."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("²", "^2").replace("±", "")
    return re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", s).replace("√", "sqrt")   # √5 → sqrt(5); √(…) stays

def read_one(p):
    """One typed answer: a number, or an equation (everything moved to one side, so it equals zero)."""
    if "=" in p:
        left, right = p.replace("==", "=").split("=", 1)
        return sp.expand(parse_expr(left, local_dict=NAMES, transformations=RULES)
                         - parse_expr(right, local_dict=NAMES, transformations=RULES))
    return parse_expr(p, local_dict=NAMES, transformations=RULES)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(p) for p in prep(s).split(",") if p.strip()]

def same_curve(got, want):
    """Two equations (each as 'something = 0') describe the same curve: one is a number times the other."""
    got, want = sp.expand(sp.sympify(got)), sp.expand(want)
    if got == 0 or not got.free_symbols <= {x, y}: return False
    mono, cw = sp.Poly(want, x, y).terms()[0]
    cg = sp.Poly(got, x, y).coeff_monomial(mono)
    return cg != 0 and sp.expand(got * cw - want * cg) == 0

def same(got, want, tol=1e-9):
    if sp.sympify(want).free_symbols: return same_curve(got, want)          # an equation
    try: return abs(complex(sp.N(sp.sympify(got) - want, 30))) < tol
    except TypeError: return False                                         # a letter, not a number

def dist(P, R): return math.hypot(R[0] - P[0], R[1] - P[1])

def branches(A, B, C, lo, hi, n=700):
    """The curve y² = Ax² + Bx + C for lo ≤ x ≤ hi, as runs of (xs, ys ≥ 0); draw each run as +ys and −ys. The
    points where y = 0 are put in exactly, so the top and bottom halves meet."""
    xs = list(np.linspace(lo, hi, n))
    if abs(A) < 1e-12: A = 0.0                             # a parabola: the x² term has cancelled
    if A: d = B * B - 4 * A * C; xs += [(-B + s * math.sqrt(d)) / (2 * A) for s in (1, -1)] if d >= 0 else []
    elif B: xs.append(-C / B)
    xs = np.array(sorted(v for v in xs if lo <= v <= hi))
    q = A * xs * xs + B * xs + C
    keep = q >= -1e-9
    runs, start = [], None
    for i, k in enumerate(list(keep) + [False]):
        if k and start is None: start = i
        if not k and start is not None:
            if i - start > 1: runs.append((xs[start:i], np.sqrt(np.clip(q[start:i], 0, None))))
            start = None
    return runs

def flat(ax, xlim, ylim):
    """A flat grid with the two axes through the origin, one unit the same both ways."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect("equal")
    ax.axhline(0, color=DIM, lw=1); ax.axvline(0, color=DIM, lw=1)

# Each question: the question, the answers in order, the working, (for rounded answers) how close counts, and (if it
# has several parts) the move that cracks each part, shown when that part is wrong.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            tol = q.get("tol", [1e-9] * len(want))
            ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · Sections of a cone

Swing a straight line round an upright axis and it sweeps out a double cone, two halves joined tip to tip. Cut it
with a flat plane. Straight across the axis, the edge of the cut is a circle. Tilt the plane a little and it's an
ellipse. Tilt it until it lies parallel to the cone's side and it's a parabola. Tilt it further, so it cuts both
halves, and it's a hyperbola. Here the cone's side makes α = 30° with its axis. Move the slider to tilt the plane:
the cut is drawn on the cone, and beside it as you'd see it looking straight at the plane. The number e = cos β /
cos α, from the plane's angle β to the axis, is the eccentricity you'll meet in chapter 5.

In [ ]:
ALPHA, H, LIFT, HINGE = 30, 4.0, 0.8, -0.3     # the cone's half-angle, its half-height; the plane turns about the
TA = math.tan(math.radians(ALPHA))            # line x = HINGE, z = LIFT (off the axis, so it never meets the tip)

def kind(beta, alpha=ALPHA):
    return "circle" if beta == 90 else "ellipse" if beta > alpha else "parabola" if beta == alpha else "hyperbola"

def ecc_cone(beta, alpha=ALPHA): return math.cos(math.radians(beta)) / math.cos(math.radians(alpha))

assert [kind(90 - t) for t in (0, 30, 60, 75, 90)] == ["circle", "ellipse", "parabola", "hyperbola", "hyperbola"]
assert round(ecc_cone(60), 2) == 0.58 and round(ecc_cone(20), 2) == 1.09 and abs(ecc_cone(30) - 1) < 1e-12   # the video's

def cut(tilt):
    """The cut curve in the plane's own coordinates (u along the tilt, v across), as runs, and the u range shown."""
    b = math.radians(90 - tilt); s, c = math.sin(b), math.cos(b)
    lo, hi, R = -9.0, 9.0, H * TA + 0.2
    if c > 1e-12: lo, hi = max(lo, (-H - LIFT) / c), min(hi, (H - LIFT) / c)      # keep |z| ≤ H
    if s > 1e-12: lo, hi = max(lo, (-R - HINGE) / s), min(hi, (R - HINGE) / s)    # and the plane beside the cone
    A = TA * TA * c * c - s * s; B = 2 * TA * TA * LIFT * c - 2 * HINGE * s; C = TA * TA * LIFT * LIFT - HINGE * HINGE
    return branches(A, B, C, lo, hi), (lo, hi), (s, c)

for t_ in (0, 20, 45, 60, 75, 90):                    # every point drawn is on the cone: x² + y² = (z tan α)²
    runs_, _, (s_, c_) = cut(t_)
    for us, vs in runs_:
        X_, Z_ = HINGE + us * s_, LIFT + us * c_
        assert np.allclose(X_ ** 2 + vs ** 2, (Z_ * TA) ** 2, atol=1e-6)

def slice_cone(tilt=0):
    beta = 90 - tilt
    runs, (lo, hi), (s, c) = cut(tilt)
    fig = plt.figure(figsize=(12, 5.2))
    ax = fig.add_axes([-0.04, -0.02, 0.5, 1.04], projection="3d")
    ax.set_facecolor(SLATE); ax.set_axis_off()
    for k in range(16):                                   # the cone's straight lines, top half and bottom half
        ph = 2 * math.pi * k / 16
        ax.plot([-H * TA * math.cos(ph), H * TA * math.cos(ph)], [-H * TA * math.sin(ph), H * TA * math.sin(ph)],
                [-H, H], color=GHOST if k % 2 else DIM, lw=1.1)
    ring = np.linspace(0, 2 * math.pi, 80)
    for z0 in (-H, H): ax.plot(H * TA * np.cos(ring), H * TA * np.sin(ring), z0, color=DIM, lw=1.4)
    V = H * TA + 0.4
    corners = [(lo, -V), (hi, -V), (hi, V), (lo, V)]
    ax.add_collection3d(Poly3DCollection([[(HINGE + u * s, v, LIFT + u * c) for u, v in corners]],
                                         facecolor=SKY, alpha=0.22, edgecolor=SKY, lw=1.2))
    for us, vs in runs:
        for sign in (1, -1): ax.plot(HINGE + us * s, sign * vs, LIFT + us * c, color=CORAL, lw=3.2)
    ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6); ax.set_zlim(-H, H); ax.set_box_aspect((1, 1, 1.45))
    ax.view_init(elev=14, azim=-62)
    p = fig.add_axes([0.5, 0.12, 0.48, 0.74]); p.set_facecolor(SLATE)
    for us, vs in runs:
        for sign in (1, -1): p.plot(us, sign * vs, color=CORAL, lw=3)
    p.set_xlim(-5.2, 4.0); p.set_ylim(-2.5, 2.5); p.set_aspect("equal"); p.set_xticks([]); p.set_yticks([])
    for side in p.spines.values(): side.set_visible(False)
    name = kind(beta)
    p.set_title(f"the cut, seen straight on: {'an' if name == 'ellipse' else 'a'} {name}", color=GOLD, fontsize=18,
                fontweight="bold", pad=12)
    rel = ("straight across the axis" if beta == 90 else f"more than α = {ALPHA}°" if beta > ALPHA else
           f"equal to α = {ALPHA}°" if beta == ALPHA else f"less than α = {ALPHA}°")
    p.set_xlabel(f"plane to axis β = {beta}°, {rel}\ne = cos β / cos α = {abs(ecc_cone(beta)):.2f}", fontsize=14,
                 linespacing=1.5)
    plt.show()

In [ ]:
interact(slice_cone, tilt=IntSlider(value=0, min=0, max=90, step=10, description="tilt (°)"));

In [ ]:
Q[1] = {"text": "Every conic is built from distances, so start there. Find the distance between the points (−3, 1) "
                "and (2, 13).",
        "answer": [13],
        "working": ["Across: 2 − (−3) = 5. Up: 13 − 1 = 12.",
                    "d = √(5² + 12²) = √(25 + 144) = √169 = 13."]}
assert dist((-3, 1), (2, 13)) == 13
practice(1)

## 2 · The parabola

A parabola is every point the same distance from a fixed point, the focus, as from a fixed line, the directrix.
With the vertex at the origin, y² = 4ax has its focus at (a, 0) and directrix x = −a; x² = 4ay has its focus at
(0, a) and directrix y = −a. A minus sign flips the curve the other way. The latus rectum, the chord through the
focus parallel to the directrix, is 4a long.

In [ ]:
Q[2] = {"text": "The parabola x² = −16y. Give its focus (two coordinates), its directrix (type only the number in "
                "y = …), and the length of its latus rectum.",
        "answer": [0, -4, 4, 16],
        "moves": ["x² = −16y is x² = −4ay with 4a = 16: the focus is on the y axis, below the vertex.",
                  "4a = 16, so a = 4, and the minus sign means the curve opens downwards.",
                  "the directrix is as far above the vertex as the focus is below it.",
                  "the latus rectum is 4a."],
        "working": ["x² = −16y has the form x² = −4ay, with 4a = 16, so a = 4.",
                    "It opens downwards (the minus sign), so the focus is (0, −4).",
                    "The directrix is the same distance on the other side: y = 4.",
                    "Latus rectum = 4a = 16.",
                    "Check: (8, −4) is on the curve (64 = −16 × −4). Its distance to the focus is 8, "
                    "and to the line y = 4 it's 4 − (−4) = 8."]}
assert 8 ** 2 == -16 * -4 and dist((8, -4), (0, -4)) == 8 == 4 - (-4)
practice(2)

## 3 · The ellipse

Push two pins into a board, loop a string round them, and pull it tight with a pencil: as the pencil goes round,
its two distances to the pins always add up to the string's length. That's an ellipse, and the pins are its foci.
If the string is 2a long and the pins are 2c apart, then a is the semi-major axis, b = √(a² − c²) the semi-minor
axis, and e = c/a the eccentricity. Move the two pins apart or together, and change the length of the string.

In [ ]:
def pins(pin1=-4.0, pin2=4.0, string=10.0):
    f1, f2 = (pin1, 0.0), (pin2, 0.0)
    m, a, c = (pin1 + pin2) / 2, string / 2, (pin2 - pin1) / 2
    fig = plt.figure(figsize=(12, 4.9))
    ax = fig.add_axes([0.03, 0.06, 0.6, 0.88]); flat(ax, (-9, 9), (-5.5, 5.5))
    p = fig.add_axes([0.63, 0.0, 0.37, 1.0]); p.axis("off")
    for nm, f in (("F₁", f1), ("F₂", f2)):
        ax.plot([f[0]], [0], "o", color=CHALK, ms=10, zorder=6); ax.text(f[0], -0.75, nm, color=CHALK, ha="center")
    if string <= pin2 - pin1 + 1e-9:
        ax.plot([pin1, pin2], [0, 0], color=CORAL, lw=3)
        p.text(0.0, 0.62, "The string is too short.\nIt must be longer than the\ngap between the pins.",
               color=GOLD, fontsize=16, va="top", linespacing=1.5)
        plt.show(); return
    b = math.sqrt(a * a - c * c)
    t = np.linspace(0, 2 * math.pi, 300)
    ax.plot(m + a * np.cos(t), b * np.sin(t), color=CHALK, lw=2.8)
    P = (m + a * math.cos(math.radians(62)), b * math.sin(math.radians(62)))          # the pencil
    total = dist(P, f1) + dist(P, f2)
    assert abs(total - string) < 1e-9                                                  # the string stays tight
    ax.plot([f1[0], P[0], f2[0]], [0, P[1], 0], color=CORAL, lw=2.4)
    ax.plot([P[0]], [P[1]], "o", color=CORAL, ms=11, zorder=7); ax.text(P[0] + 0.2, P[1] + 0.35, "P", color=CORAL)
    ax.plot([m, m + a], [0, 0], color=GOLD, lw=4, solid_capstyle="butt", zorder=4)     # a
    ax.plot([m, m], [0, b], color=MINT, lw=4, solid_capstyle="butt", zorder=4)         # b
    ax.plot([m, f2[0]], [-0.18, -0.18], color=SKY, lw=4, solid_capstyle="butt", zorder=4)   # c
    p.text(0.0, 0.92, f"PF₁ + PF₂ = {dist(P, f1):.2f} + {dist(P, f2):.2f}\n          = {total:.2f}, the string",
           color=CORAL, fontsize=14, va="top", family="monospace")
    p.text(0.0, 0.66, f"a = string ÷ 2       = {a:.2f}", color=GOLD, fontsize=15, family="monospace")
    p.text(0.0, 0.55, f"c = pin gap ÷ 2      = {c:.2f}", color=SKY, fontsize=15, family="monospace")
    p.text(0.0, 0.44, f"b = √(a² − c²)       = {b:.2f}", color=MINT, fontsize=15, family="monospace")
    p.text(0.0, 0.29, f"e = c/a = {c / a:.2f}", color=CHALK, fontsize=20, fontweight="bold")
    p.text(0.0, 0.17, "a circle: the pins together" if c == 0 else
           "close to a circle" if c / a < 0.4 else "long and thin" if c / a > 0.85 else "", color=DIM, fontsize=14)
    plt.show()

In [ ]:
interact(pins, pin1=FloatSlider(value=-4, min=-7, max=0, step=0.5, description="pin F₁"),
         pin2=FloatSlider(value=4, min=0, max=7, step=0.5, description="pin F₂"),
         string=FloatSlider(value=10, min=2, max=16, step=0.5, description="string"));

In [ ]:
Q[3] = {"text": "The ellipse 16x² + 25y² = 400. Its foci are (±c, 0): give c, then the eccentricity e, then the length "
                "of the latus rectum.",
        "answer": [3, r(3, 5), r(32, 5)],
        "moves": ["divide by 400 first: x²/25 + y²/16 = 1, so a² = 25 and b² = 16; then c² = a² − b².",
                  "e = c/a, with a = 5.", "the latus rectum is 2b²/a."],
        "working": ["Divide by 400: x²/25 + y²/16 = 1, so a = 5 and b = 4.",
                    "c² = a² − b² = 25 − 16 = 9, so c = 3 and the foci are (±3, 0).",
                    "e = c/a = 3/5 = 0.6.",
                    "Latus rectum = 2b²/a = 2 × 16 / 5 = 32/5 = 6.4."]}
assert 400 // 16 == 25 and 400 // 25 == 16 and 25 - 16 == 9 and F(2 * 16, 5) == F(32, 5)
assert abs(dist((3, 3.2), (3, 0)) + dist((3, 3.2), (-3, 0)) - 10) < 1e-12 and 16 * 9 + 25 * F(16, 5) ** 2 == 400   # (c, b²/a) is on it
practice(3)

## 4 · The hyperbola

A hyperbola is every point whose distances to two foci differ by the same amount, 2a. With the foci at (±c, 0),
x²/a² − y²/b² = 1 with c² = a² + b². Far from the centre it hugs its two asymptotes, y = ±(b/a)x. Its eccentricity
e = c/a is more than 1, and its latus rectum is 2b²/a, as for the ellipse.

In [ ]:
Q[4] = {"text": "The hyperbola x²/16 − y²/9 = 1. Give c (its foci are (±c, 0)), its eccentricity, the slope of its "
                "uphill asymptote, and the length of its latus rectum.",
        "answer": [5, r(5, 4), r(3, 4), r(9, 2)],
        "moves": ["for a hyperbola c² = a² + b² (a plus, not a minus).", "e = c/a, with a = 4.",
                  "the asymptotes are y = ±(b/a)x.", "the latus rectum is 2b²/a."],
        "working": ["a² = 16 and b² = 9, so a = 4 and b = 3.",
                    "c² = a² + b² = 16 + 9 = 25, so c = 5 and the foci are (±5, 0).",
                    "e = c/a = 5/4 = 1.25 (more than 1, as it must be).",
                    "Asymptotes: y = ±(3/4)x, so the uphill one has slope 3/4.",
                    "Latus rectum = 2b²/a = 2 × 9 / 4 = 9/2 = 4.5."]}
assert 16 + 9 == 25 and F(5, 4) > 1 and F(2 * 9, 4) == F(9, 2)
assert abs(abs(dist((5, 2.25), (-5, 0)) - dist((5, 2.25), (5, 0))) - 8) < 1e-12 and 5 ** 2 / 16 - 2.25 ** 2 / 9 == 1
practice(4)

## 5 · Focus, directrix and eccentricity

One rule makes all three curves: the distance to a focus S is e times the distance to a directrix, PS = e · PM. With
e < 1 the curve is an ellipse, with e = 1 a parabola, and with e > 1 a hyperbola. Here the focus sits at the origin
and the directrix is the upright line x = 3; neither moves. Move the slider for e: the curve changes, and the point
P shows its two distances as it goes.

In [ ]:
D5 = 3                                                     # the directrix x = 3; the focus S = (0, 0)

def ecc_kind(e): return "parabola" if abs(e - 1) < 1e-9 else "ellipse" if e < 1 else "hyperbola"

def ecc_point(e, th=60):
    """The point of the curve at angle th from the focus: r = ed / (1 + e cos th)."""
    rr = e * D5 / (1 + e * math.cos(math.radians(th)))
    return rr * math.cos(math.radians(th)), rr * math.sin(math.radians(th))

for e_ in (0.3, 0.5, 1.0, 1.5, 2.0):                      # PS = e · PM, at the moving point
    P_ = ecc_point(e_); assert abs(dist(P_, (0, 0)) - e_ * (D5 - P_[0])) < 1e-12

def eccentricity(e=0.5):
    e = round(e, 6)
    fig = plt.figure(figsize=(12, 5.0))
    ax = fig.add_axes([0.04, 0.08, 0.54, 0.86]); flat(ax, (-8, 12), (-5.5, 5.5)); ax.set_xticks(range(-8, 13, 2))
    p = fig.add_axes([0.62, 0.0, 0.38, 1.0]); p.axis("off")
    for xs, ys in branches(e * e - 1, -2 * e * e * D5, e * e * D5 * D5, -8, 12):      # x² + y² = e²(3 − x)²
        ax.plot(xs, ys, color=CHALK, lw=2.8); ax.plot(xs, -ys, color=CHALK, lw=2.8)
    ax.axvline(D5, color=SKY, lw=2.4, ls="--"); ax.text(D5 + 0.2, -5.1, "directrix x = 3", color=SKY, fontsize=12)
    ax.plot([0], [0], "o", color=GOLD, ms=11, zorder=6); ax.text(-0.3, -0.75, "S", color=GOLD, ha="right")
    P = ecc_point(e)
    PS, PM = dist(P, (0, 0)), D5 - P[0]
    ax.plot([0, P[0]], [0, P[1]], color=MINT, lw=3); ax.plot([P[0], D5], [P[1], P[1]], color=SKY, lw=3)
    ax.plot([P[0]], [P[1]], "o", color=CORAL, ms=12, zorder=7); ax.text(P[0] - 0.3, P[1] + 0.35, "P", color=CORAL, ha="right")
    p.text(0.0, 0.88, f"e = {e:.2f}", color=CHALK, fontsize=24, fontweight="bold")
    p.text(0.0, 0.74, f"{'an' if ecc_kind(e) == 'ellipse' else 'a'} {ecc_kind(e)}", color=GOLD, fontsize=24, fontweight="bold")
    p.text(0.0, 0.56, f"PS = {PS:.2f}", color=MINT, fontsize=17, family="monospace")
    p.text(0.0, 0.46, f"PM = {PM:.2f}", color=SKY, fontsize=17, family="monospace")
    p.text(0.0, 0.32, f"PS ÷ PM = {PS / PM:.2f} = e", color=CORAL, fontsize=17, family="monospace")
    p.text(0.0, 0.18, "e < 1: ellipse  ·  e = 1: parabola\ne > 1: hyperbola", color=DIM, fontsize=13, va="top",
           linespacing=1.5)
    plt.show()

In [ ]:
interact(eccentricity, e=FloatSlider(value=0.5, min=0.2, max=2.0, step=0.1, description="e"));

In [ ]:
Q[5] = {"text": "A conic has its focus at (2, 0), its directrix x = 8, and eccentricity e = 1/2. Find its equation, in "
                "any form.",
        "answer": [3 * x ** 2 + 4 * y ** 2 - 48],
        "working": ["PS = e · PM: √((x − 2)² + y²) = ½ |x − 8|.",
                    "Square both sides: (x − 2)² + y² = ¼(x − 8)².",
                    "Times 4: 4x² − 16x + 16 + 4y² = x² − 16x + 64.",
                    "So 3x² + 4y² = 48, or x²/16 + y²/12 = 1: an ellipse, as e < 1 says.",
                    "Check: the point (4, 0) is on it. It's 2 from the focus and 4 from the directrix, and 2 = ½ × 4."]}
assert sp.expand(4 * ((x - 2) ** 2 + y ** 2) - (x - 8) ** 2) == 3 * x ** 2 + 4 * y ** 2 - 48
assert same_curve(read("x^2/16 + y^2/12 = 1")[0], Q[5]["answer"][0]) and dist((4, 0), (2, 0)) == F(1, 2) * (8 - 4)
practice(5)

## 6 · Parametric equations and tangents

One number t can walk a point round a conic: (at², 2at) on the parabola y² = 4ax, (a cos θ, b sin θ) on an ellipse.
The tangent to y² = 4ax at the point with parameter t is ty = x + at². At a point (x₁, y₁) of an ellipse,
the tangent is xx₁/a² + yy₁/b² = 1, and of a hyperbola xx₁/a² − yy₁/b² = 1.

In [ ]:
Q[6] = {"text": "On the parabola y² = 12x, find the point with parameter t = 2 (two coordinates). Then give the "
                "tangent at that point, as an equation in any form.",
        "answer": [12, 12, x - 2 * y + 12],
        "moves": ["4a = 12, so a = 3; the point is (at², 2at).", "4a = 12, so a = 3; the point is (at², 2at).",
                  "the tangent at t is ty = x + at²."],
        "working": ["y² = 12x is y² = 4ax with a = 3.",
                    "t = 2: (at², 2at) = (3 × 4, 2 × 3 × 2) = (12, 12). Check: 12² = 144 = 12 × 12.",
                    "Tangent: ty = x + at², so 2y = x + 12 (or y = x/2 + 6).",
                    "Check: put x = 2y − 12 into y² = 12x: y² − 24y + 144 = 0, which is (y − 12)² = 0.",
                    "One repeated answer, y = 12: the line touches the curve at just one point."]}
assert 12 ** 2 == 12 * 12 and 2 * 12 == 12 + 12
assert sp.expand(y ** 2 - 12 * (2 * y - 12)) == sp.expand((y - 12) ** 2)
practice(6)

## 7 · The general equation of a conic

Any conic can be written Ax² + Bxy + Cy² + Dx + Ey + F = 0. The discriminant B² − 4AC names it: negative for an
ellipse (or a circle), zero for a parabola, positive for a hyperbola. Then completing the square in x and in y finds
its centre and its size.

In [ ]:
Q[7] = {"text": "The conic x² + 4y² − 6x + 16y + 21 = 0. Give its discriminant B² − 4AC, then its centre (two "
                "coordinates), then its semi-axes a and b (the longer first).",
        "answer": [-16, 3, -2, 2, 1],
        "moves": ["A = 1, B = 0, C = 4.", "complete the square: x² − 6x = (x − 3)² − 9.",
                  "complete the square: 4y² + 16y = 4(y + 2)² − 16.",
                  "divide by the number left on the right, to get (x − 3)²/a² + (y + 2)²/b² = 1.",
                  "divide by the number left on the right, to get (x − 3)²/a² + (y + 2)²/b² = 1."],
        "working": ["A = 1, B = 0, C = 4, so B² − 4AC = 0 − 16 = −16 < 0: an ellipse.",
                    "x² − 6x = (x − 3)² − 9, and 4y² + 16y = 4(y² + 4y) = 4(y + 2)² − 16.",
                    "So (x − 3)² − 9 + 4(y + 2)² − 16 + 21 = 0, which is (x − 3)² + 4(y + 2)² = 4.",
                    "Divide by 4: (x − 3)²/4 + (y + 2)²/1 = 1.",
                    "Centre (3, −2); a = 2 across and b = 1 up and down."]}
assert all(xv * xv + 4 * yv * yv - 6 * xv + 16 * yv + 21 == (xv - 3) ** 2 + 4 * (yv + 2) ** 2 - 4
           for xv, yv in ((0, 0), (1, 2), (-3, 5), (2, -3)))
assert 0 ** 2 - 4 * 1 * 4 == -16 and 5 ** 2 + 4 * 2 ** 2 - 6 * 5 + 16 * -2 + 21 == 0   # (5, −2), a to the right, is on it
practice(7)

## Sources
NCERT Mathematics Class XI, ch. 10 (Conic Sections); OpenStax, Precalculus 2e, ch. 10 (Analytic Geometry) and
College Algebra 2e, ch. 8. All retrieved 2026-10-31. Every question is our own.